In [1]:
import os

os .environ ['US_TIGER_SHAPEFILE_PATH']='/Users/aaronspaulding/Documents/PycharmProjects/outagescraping/TIGER/tl_2020_us_county/tl_2020_us_county.shp'
os .environ ['CANADA_CENSUS_DIVISIONS_PATH']='/Users/aaronspaulding/Documents/PycharmProjects/outagescraping/CanadaBoundaries/lcd_000b21a_e/lcd_000b21a_e.shp'
os .environ ['DATA_CACHE_PATH']='/Users/aaronspaulding/Documents/PycharmProjects/research_summer_2025/data_cache/'
os .environ ['PYTORCH_ENABLE_MPS_FALLBACK']='1'


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import matplotlib .pyplot as plt

import contextily as ctx

from tqdm import tqdm

from research_spring_2026 import load_us_counties
from research_spring_2026 import get_selected_regions
from research_spring_2026 import generate_h3_grid_from_gdf
from research_spring_2026 import get_parent_cells

import torch

import pyro
import pyro .distributions as dist
from pyro .nn import PyroParam

from pyro .infer import SVI ,Trace_ELBO
from pyro .optim import ClippedAdam


In [3]:
counties =get_selected_regions ()
h3_cells =generate_h3_grid_from_gdf (counties ,use_cached_file =True )


In [4]:
h3_map =(
h3_cells [['index']]
.drop_duplicates ()
.sort_values ('index')
.reset_index (drop =True )
.reset_index ()
.rename (columns ={'index':'index','level_0':'h3_code'})
)

h3_map ['h3_code']=h3_map ['h3_code'].astype ('uint32')


In [5]:
relevant_storms =gpd .read_feather ('data_cache/relevant_storm_tracks.feather')
relevant_storms =relevant_storms [::-1 ].reset_index (drop =True )


## Static Variables

- H3 index (to merge with temporal variables)
- State, County, and Tract Assignments
- Length of linmes in each cell
- Length of roads in each cell for each year
- Tree Canopy Cover


In [9]:
static_variables_fips_and_census =pd .read_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'static_variables_fips_and_census.feather')
static_variables_length_of_lines =pd .read_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'static_variables_length_of_lines.feather')
static_variables_length_of_roads =pd .read_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'static_variables_length_of_roads.feather')
static_variables_TCC =pd .read_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'static_variables_TCC.feather')
static_variables_elevation =pd .read_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'static_variables_elevation.feather')
static_variables_utility_information =pd .read_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'static_variables_utility_locations.feather')
static_variables_svi =pd .read_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'static_variables_svi.feather')
static_variables_tree_canopy_height =pd .read_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'static_variables_tree_canopy_height.feather')


In [10]:
h3_cells_with_static_variables =h3_cells .copy ()
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (h3_map ,on ='index',how ='left')
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (static_variables_fips_and_census ,on ='index',how ='left')
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (static_variables_length_of_lines ,on ='index',how ='left')
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (static_variables_length_of_roads ,on ='index',how ='left')
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (static_variables_TCC ,on ='index',how ='left')
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (static_variables_elevation ,on ='index',how ='left')
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (static_variables_utility_information ,on ='index',how ='left')
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (static_variables_svi ,on ='index',how ='left')
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (static_variables_tree_canopy_height ,on ='index',how ='left')


In [11]:
h3_cells_with_static_variables ['FULL_COUNTY_FIPS']=h3_cells_with_static_variables ['STATEFP']+h3_cells_with_static_variables ['COUNTYFP']

h3_cells_with_static_variables ['FULL_TRACT_FIPS']=h3_cells_with_static_variables ['STATEFP']+h3_cells_with_static_variables ['COUNTYFP']+h3_cells_with_static_variables ['TRACTFP']


In [12]:
h3_county_map =(
h3_cells_with_static_variables [['FULL_COUNTY_FIPS']]
.drop_duplicates (subset =['FULL_COUNTY_FIPS'])
.sort_values ('FULL_COUNTY_FIPS')
.reset_index (drop =True )
.reset_index ()
.rename (columns ={'index':'county_map','FULL_COUNTY_FIPS':'FULL_COUNTY_FIPS'})
)
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (h3_county_map ,on ='FULL_COUNTY_FIPS',how ='left')


In [13]:
h3_state_map =(
h3_cells_with_static_variables [['STATEFP']]
.drop_duplicates (subset =['STATEFP'])
.sort_values ('STATEFP')
.reset_index (drop =True )
.reset_index ()
.rename (columns ={'index':'state_map','STATEFP':'STATEFP'})
)
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (h3_state_map ,on ='STATEFP',how ='left')


In [14]:
h3_county_map =(
h3_cells_with_static_variables [['FULL_TRACT_FIPS']]
.drop_duplicates (subset =['FULL_TRACT_FIPS'])
.sort_values ('FULL_TRACT_FIPS')
.reset_index (drop =True )
.reset_index ()
.rename (columns ={'index':'tract_map','FULL_TRACT_FIPS':'FULL_TRACT_FIPS'})
)
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (h3_county_map ,on ='FULL_TRACT_FIPS',how ='left')


In [15]:
number_of_customers_tracked_in_each_county_path ='/Users/aaronspaulding/Documents/PycharmProjects/research_summer_2025/data_cache/number_of_customers_tracked_in_each_county.feather'
number_of_customers_tracked_in_each_county =pd .read_feather (number_of_customers_tracked_in_each_county_path )
number_of_customers_tracked_in_each_county .columns =['FULL_COUNTY_FIPS','COUNTY_UTILITY_CUSTOMERS']


In [16]:
h3_cells_with_static_variables =h3_cells_with_static_variables .merge (number_of_customers_tracked_in_each_county ,on ='FULL_COUNTY_FIPS',how ='left')
h3_cells_with_static_variables ['COUNTY_UTILITY_CUSTOMERS']=h3_cells_with_static_variables ['COUNTY_UTILITY_CUSTOMERS'].fillna (-1 )


In [17]:
h3_cells_with_static_variables ['tree_height_min']=h3_cells_with_static_variables ['tree_height_min'].fillna (0.0 )
h3_cells_with_static_variables ['tree_height_max']=h3_cells_with_static_variables ['tree_height_max'].fillna (0.0 )
h3_cells_with_static_variables ['tree_height_mean']=h3_cells_with_static_variables ['tree_height_mean'].fillna (0.0 )
h3_cells_with_static_variables ['tree_height_median']=h3_cells_with_static_variables ['tree_height_median'].fillna (0.0 )
h3_cells_with_static_variables ['tree_height_std']=h3_cells_with_static_variables ['tree_height_std'].fillna (0.0 )


In [18]:
h3_cells_with_static_variables .to_feather (Path (os .environ .get ('DATA_CACHE_PATH'))/'aggregated_static_variables.feather')


## Temporal Variables

- Wind speed
- Gust speed
- Number of outages in each grid cell


In [19]:
temporal_variables_cache =Path ('/Users/aaronspaulding/Documents/PycharmProjects/research_summer_2026/data_cache/temporal_variables_cache_for_each_storm/')
files_in_temporal_variables_cache =list (temporal_variables_cache .glob ('*.feather'))

hourly_outages_files =[f for f in files_in_temporal_variables_cache if f .name .startswith ('outages_hourly_')]
storm_outages_files =[f for f in files_in_temporal_variables_cache if f .name .startswith ('outages_storm_')]
weather_outages_files =[f for f in files_in_temporal_variables_cache if f .name .startswith ('weather_')]

def extract_storm_id (f ,prefix ):
    return f .name .replace (prefix ,'').replace ('.feather','')

hourly_storm_ids ={extract_storm_id (f ,'outages_hourly_')for f in hourly_outages_files }
storm_storm_ids ={extract_storm_id (f ,'outages_storm_')for f in storm_outages_files }
weather_storm_ids ={extract_storm_id (f ,'weather_')for f in weather_outages_files }

valid_storm_ids =list (set (hourly_storm_ids &storm_storm_ids &weather_storm_ids ))

relevant_storms =relevant_storms [relevant_storms .SID .isin (valid_storm_ids )].reset_index (drop =True )


In [20]:
from shapely .geometry import Point

def lines_to_points (gdf :gpd .GeoDataFrame )->gpd .GeoDataFrame :
    """Explode a GeoDataFrame of LineStrings into individual Point rows."""
    records =[]
    for idx ,row in gdf .iterrows ():
        for x ,y in row .geometry .coords :
            rec =row .to_dict ()
            rec ['geometry']=Point (x ,y )
            records .append (rec )
    return gpd .GeoDataFrame (records ,crs =gdf .crs )

relevant_storms_points =lines_to_points (relevant_storms )


In [21]:
import faiss

h3_cells_projected =h3_cells_with_static_variables .to_crs ('EPSG:6933')
relevant_storms_points_projected =relevant_storms_points .to_crs ('EPSG:6933')

h3_cells_projected ['center']=h3_cells_projected .centroid
h3_cells_projected ['x']=h3_cells_projected ['center'].x
h3_cells_projected ['y']=h3_cells_projected ['center'].y
cells_projected_centers =np .array (h3_cells_projected [['x','y']],dtype =np .float32 )

relevant_storms_points_projected ['center']=relevant_storms_points_projected .centroid
relevant_storms_points_projected ['x']=relevant_storms_points_projected ['center'].x
relevant_storms_points_projected ['y']=relevant_storms_points_projected ['center'].y


In [28]:
for i ,row in relevant_storms .iterrows ():

    storm_id =row ['SID']
    storm_name =row ['NAME']
    storm_start =row ['datetime_min'].floor ('h')
    storm_end =row ['datetime_max'].ceil ('h')+pd .to_timedelta (1 ,'d')

    print (i ,storm_name )

    hourly_outages_file_path =temporal_variables_cache /('outages_hourly_'+storm_id +'.feather')
    storm_outages_file_path =temporal_variables_cache /('outages_storm_'+storm_id +'.feather')
    weather_file_path =temporal_variables_cache /('weather_'+storm_id +'.feather')

    hourly_outages =pd .read_feather (hourly_outages_file_path )
    hourly_outages ['hour']=(
    (hourly_outages ['datetime']-hourly_outages ['datetime'].min ()).dt .total_seconds ()//3600 ).astype (int )

    weather_data =pd .read_feather (weather_file_path )

    weather_data =weather_data .merge (
    hourly_outages [['outage_count_for_this_hour','hour','h3_code']],
    on =['hour','h3_code'],
    how ='left',
    )
    weather_data ['outage_count_for_this_hour']=weather_data ['outage_count_for_this_hour'].astype ('float32')

    weather_data =weather_data [weather_data ['h3_code'].isin (h3_cells_with_static_variables ['h3_code'])].reset_index (drop =True )

    aggregated_file_path =temporal_variables_cache /('aggregated_temporal_'+storm_id +'.feather')
    weather_data .to_feather (aggregated_file_path )

    break


0 ISAIAS


# Temporal: Add in flood data


In [35]:
base_path ='/Users/aaronspaulding/Documents/PycharmProjects/research_spring_2026/data_cache/temporal_variables_cache_for_each_storm/'

all_files =os .listdir (base_path )
agg_files =[f for f in all_files if f .startswith ('aggregated_temporal_')and f .endswith ('.feather')]

for agg_file in agg_files :
    storm_id =agg_file .split ('_')[2 ][:-len ('.feather')]

    flood_data_file_path =base_path +f'flooding_{storm_id}.feather'

    if not os .path .exists (flood_data_file_path ):
        continue

    aggregated_data =pd .read_feather (base_path +agg_file )

    flood_data =pd .read_feather (flood_data_file_path )
    flood_data ['flooded']=flood_data ['flooded'].astype (int )

    aggregated_data =aggregated_data .merge (flood_data [['h3_code','flooded']],how ='left',on ='h3_code')
    aggregated_data .to_feather (base_path +agg_file )


In [33]:
aggregated_data


NameError: name 'aggregated_data' is not defined

In [73]:
aggregated_data


,h3_code,wind_speed,gust_speed,accumulated_precipitation,hour,outage_count_for_this_hour
0,2,5.593828,15.720585,1.26,0,NaN
1,6,5.195477,14.970585,1.45,0,NaN
2,14,6.275070,16.095585,1.93,0,NaN
3,15,5.599669,15.408085,2.27,0,NaN
4,16,6.275070,16.095585,1.93,0,NaN
...,...,...,...,...,...,...
14244115,2990177,0.661742,1.261188,0.00,39,NaN
14244116,2990178,0.661742,1.261188,0.00,39,NaN
14244117,2990179,0.771814,1.323688,0.00,39,NaN
14244118,2990184,0.771814,1.323688,0.00,39,NaN


In [ ]:
flood_data_file_path


In [ ]:
agg_files


In [ ]:
cells_projected_centers .shape


In [ ]:
h3_cells_with_static_variables [cell_to_keep_idx ].plot ()


In [ ]:
h3_cells_with_static_variables [cell_to_keep_idx ].shape


In [ ]:
counties_projected =counties .to_crs ('EPSG:6933')
ibtracs_projected =ibtracs .to_crs ('EPSG:6933')

counties_projected ['center']=counties_projected .centroid
counties_projected ['x']=counties_projected ['center'].x
counties_projected ['y']=counties_projected ['center'].y
cells_projected_centers =np .array (counties_projected [['x','y']],dtype =np .float32 )

ibtracs_projected ['center']=ibtracs_projected .centroid
ibtracs_projected ['x']=ibtracs_projected ['center'].x
ibtracs_projected ['y']=ibtracs_projected ['center'].y
ibtracs_projected_centers =np .array (ibtracs_projected [['x','y']],dtype =np .float32 )

faiss_index =faiss .IndexFlatL2 (2 )
faiss_index .add (cells_projected_centers )
distances ,index =faiss_index .search (ibtracs_projected_centers ,1 )

ibtracs ['distance_to_closest_point']=np .sqrt (distances )


In [ ]:
h3_cells_with_static_variables .geometry


In [ ]:
weather_data


In [ ]:
list (row .geometry .coords )


In [ ]:
weather_data .shape


In [ ]:
a =weather_data .groupby ('hour').agg ({
'gust_speed':np .max ,
'outage_count_for_this_hour':np .nanmax
}
)


In [ ]:
plt .plot (a ['gust_speed'])


In [ ]:
weather_data
